# MSADS 599 - Modeling Part 1 - Classification - Random Forest

## Setup

Import Packages

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, login
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler

# This shows all columns when analyzing the data
pd.set_option('display.max_columns', None)

Connect to HuggingFace and Import Data

In [4]:
# Connection to HuggingFace via API token
login(token="HF_TOKEN")

In [6]:
# Import training data
train_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_train/part1_train-00000-of-00001.parquet",
    repo_type = "dataset"
)

train_df = pd.read_parquet(train_path)
print(train_df.shape)

part1_train/part1_train-00000-of-00001.p(…):   0%|          | 0.00/1.32M [00:00<?, ?B/s]

(67085, 73)


In [7]:
# Import testing data
test_path = hf_hub_download(
    repo_id = "ADS599-Capstone/modeling_data",
    filename = "part1_test/part1_test-00000-of-00001.parquet",
    repo_type = "dataset"
)

test_df = pd.read_parquet(test_path)
print(test_df.shape)

part1_test/part1_test-00000-of-00001.par(…):   0%|          | 0.00/381k [00:00<?, ?B/s]

(16882, 73)


## Modeling

Clean and Set the Target Variable

In [8]:
icu_labels = {'ED_DIRECT_ICU', 'ED_WARD_ICU'}

train_df['target'] = train_df['cohort_label'].apply(lambda x: 1 if x in icu_labels else 0)
test_df['target'] = test_df['cohort_label'].apply(lambda x: 1 if x in icu_labels else 0)

# Fit ONLY on train, transform both
le = LabelEncoder()
train_df['target'] = le.fit_transform(train_df['target'])
test_df['target'] = le.transform(test_df['target'])  # ← transform only, not fit_transform

class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(class_mapping)

{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}


In [9]:
# Split the target variable from the test and train datasets

# Training Set
X_train = train_df.drop(columns=['cohort_label', 'target'])
y_train = train_df['target']

# Testing Set
X_test = test_df.drop(columns=['cohort_label', 'target'])
y_test = test_df['target']

# Sanity Check
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (67085, 72), y_train: (67085,)
X_test: (16882, 72), y_test: (16882,)


Train the Model

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=35,
    n_jobs=-1,
    max_features='sqrt' # Preferred log2
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=35)

In [11]:
rf_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=35,
    n_jobs=-1,
)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_grid,
    n_iter=30,              # Number of random combinations to try
    cv=5,
    scoring='f1_weighted',  # Weighted for multiclass
    n_jobs=-1,
    random_state=35,
    verbose=2,
)

rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
print("Best RF score:", rf_search.best_score_)
rf_model = rf_search.best_estimator_  # Drop-in replacement for your original rf_model

# Best Random Forest Hyperparameters
# {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30}

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best RF params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
Best RF score: 0.8322637699109509


Evaluate the Model

In [12]:
target_names = ['Discharge', 'ICU']  # index 0=Discharge, 1=ICU

y_pred = rf_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

TypeError: object of type 'numpy.int64' has no len()

Feature Importance - Random Forest

In [ ]:
importances = rf_model.feature_importances_
feature_names = X_train.columns
indices = np.argsort(importances)[::-1][:20] # Top 20 features

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(20), importances[indices][::-1], color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels(feature_names[indices][::-1])
ax.set_title("Top 20 Most Importance Features")
ax.set_xlabel("Feature Importance (GINI Impurity)")
plt.tight_layout()
plt.show()